In [21]:
from langchain_openai import ChatOpenAI 
import os
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(model ="gpt-5-mini") 


### Tools

In [11]:
from langchain.tools import tool

In [12]:
@tool # Decorator to mark this function as a tool
def tool_duckduckgo_search(query: str) -> str:
    """Use this tool when you need to answer questions about current events or general knowledge questions."""
   
    from langchain_community.tools import DuckDuckGoSearchRun

    search = DuckDuckGoSearchRun()

    response = search.invoke(query)
    return response

In [15]:
tool_duckduckgo_search.invoke("What is the capital of India?")

"India, officially the Republic of India, [j][20] is a country in South Asia. It is the seventh-largest country by area; the most populous country since 2023; [21] and, since its independence in 1947, the world's most populous democracy. [22][23][24] Bounded by the Indian Ocean on the south, the Arabian Sea on the southwest, and the Bay of Bengal on the southeast, it shares land borders with ... Updated list of States and Capitals of India, All 28 states and 8 Union Territories of India with capitals. Includes formation dates, recent changes. Get interesting information and facts about India's capital city, New Delhi -- its history, government, geography, climate and urban structure. Complete states and capitals of India list 2026 — all 28 states, 8 UTs, capitals, languages & facts. Updated. Essential for UPSC, SSC, Banking & competitive exams. Revise now. Discover whether Delhi or New Delhi is the true capital of India. Understand their historical roots, administrative differences, an

In [17]:
@tool # Decorator to mark this function as a tool
def tool_wikipedia_search(query: str) -> str:
    """Use this tool when you need to answer questions about persons, places etc."""
    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper
    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    response = wikipedia.run(query)
    return response

In [18]:
tool_wikipedia_search.invoke("Who is the president of USA?")

'Page: List of presidents of the United States\nSummary: The president of the United States is the head of state and head of government of the United States, indirectly elected to a four-year term via the Electoral College. Under the U.S. Constitution, the officeholder leads the executive branch of the federal government and is the commander-in-chief of the United States Armed Forces.\nThe first president, George Washington, won a unanimous vote of the Electoral College. The incumbent president is Donald Trump, who assumed office on January 20, 2025. Since the office was established in 1789, 45 men have served in 47 presidencies. The discrepancy is due to the nonconsecutive terms of Grover Cleveland (counted as the 22nd and 24th president) and Trump (counted as the 45th and 47th president).\nThe presidency of William Henry Harrison, who died 31 days after taking office in 1841, was the shortest in American history. Franklin D. Roosevelt served the longest, over twelve years, before dyi

In [19]:
@tool
def tool_arxiv_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about scientific papers or research topics. """

    from langchain_community.tools import ArxivQueryRun
    from langchain_community.utilities import ArxivAPIWrapper

    # 1. Initialize the arXiv API wrapper
    arxiv_wrapper = ArxivAPIWrapper(
        top_k_results=3,       # Number of papers to retrieve
        doc_content_chars_max=2000  # Max characters per document
    )

    # 2. Create the arXiv tool
    arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)

    # 3. Use the tool directly
    result = arxiv_tool.run(query)

    print(result)

In [20]:
tool_arxiv_search.invoke("What are the latest research papers on deep learning?")

Published: 2023-01-03
Title: Deep Learning and Computational Physics (Lecture Notes)
Authors: Deep Ray, Orazio Pinti, Assad A. Oberai
Summary: These notes were compiled as lecture notes for a course developed and taught at the University of the Southern California. They should be accessible to a typical engineering graduate student with a strong background in Applied Mathematics.
  The main objective of these notes is to introduce a student who is familiar with concepts in linear algebra and partial differential equations to select topics in deep learning. These lecture notes exploit the strong connections between deep learning algorithms and the more conventional techniques of computational physics to achieve two goals. First, they use concepts from computational physics to develop an understanding of deep learning algorithms. Not surprisingly, many concepts in deep learning can be connected to similar concepts in computational physics, and one can utilize this connection to better un

In [39]:
@tool
def tool_personal_info(query: str) -> str:
    """Use this tool when you need to answer questions about personal information."""
    infos = [{
        "name": "John Doe",
        "age": 30,
        "Occupation": "Software Engineer"
    },
    {
        "name": "Jane Smith",
        "age": 25,
        "Occupation": "Data Scientist"
    }]
    for info in infos:
        if query.lower() in info["name"].lower():
            return f"{info['name']} is a {info['age']} year old and works as a {info['Occupation']}."
    return "No information found."

In [40]:
tool_personal_info.invoke("John Doe")

'John Doe is a 30 year old and works as a Software Engineer.'

### Binding Tools

In [41]:
toolkit = [
    tool_duckduckgo_search,
    tool_wikipedia_search,
    tool_arxiv_search,
    tool_personal_info
    
]

In [42]:
llm_bind = llm.bind_tools(toolkit) #.bind_tools() is function that binds the tools to the llm. It takes a list of tools as input and returns a new llm that has access to those tools.

In [43]:
llm_bind.invoke("What is the age of John Doe? Make tool calls if necessary to answer the question.")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 249, 'total_tokens': 339, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVyU2ESThCpB3uILxVnQHg5RKues4', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da06d-b4e5-7b62-b54a-00bfe11065a7-0', tool_calls=[{'name': 'tool_wikipedia_search', 'args': {'query': 'John Doe'}, 'id': 'call_Y9uqPMd3is6FiqD6SxjdLGkO', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 249, 'output_tokens': 90, 'total_tokens': 339, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 64}})